# Laboratorio 8 — Veiumtum (Blackjack/21) con Q-learning

Implementacion del juego Veiumtum (Blackjack) como entorno personalizado de gymnasium, resuelto con Q-learning.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

## 1. Funciones del entorno Veiumtum

- **Accion 0**: Plantarse (Stick)
- **Accion 1**: Pedir (Hit)
- **Observacion**: (suma_jugador, carta_crupier, as_utilizable)
- **Baraja**: infinita (con reemplazo), figuras=10, As=1 u 11
- **Crupier**: pide hasta 17+
- **Recompensas**: +1 victoria, -1 derrota, 0 empate

In [ ]:
def crear_entorno():
    return {
        'jugador': [min(10, np.random.randint(1, 14)) for _ in range(2)],
        'crupier': [min(10, np.random.randint(1, 14)) for _ in range(2)],
    }


def valor_mano(mano):
    total = sum(mano)
    ases = mano.count(1)
    while total + 10 <= 21 and ases > 0:
        total += 10
        ases -= 1
    return total


def as_utilizable(mano):
    return 1 if (1 in mano and valor_mano(mano) <= 11) else 0


def observacion(estado):
    return (valor_mano(estado['jugador']), estado['crupier'][0], as_utilizable(estado['jugador']))


def paso(estado, accion):
    if accion == 1:
        estado['jugador'].append(min(10, np.random.randint(1, 14)))
        if valor_mano(estado['jugador']) > 21:
            return observacion(estado), -1.0, True

    while valor_mano(estado['crupier']) < 17:
        estado['crupier'].append(min(10, np.random.randint(1, 14)))

    pv = valor_mano(estado['jugador'])
    dv = valor_mano(estado['crupier'])

    if dv > 21 or pv > dv:
        reward = 1.0
    elif pv == dv:
        reward = 0.0
    else:
        reward = -1.0

    return observacion(estado), reward, True


def reset():
    estado = crear_entorno()
    return estado, observacion(estado)

## 2. Entrenamiento con Q-learning

In [ ]:
def entrenar(episodios=50000, lr=0.1, gamma=0.95, eps=1.0, eps_min=0.01, eps_decay=0.999):
    Q, rewards, wins = {}, [], 0

    for ep in range(episodios):
        estado, obs = reset()
        done = False
        total_reward = 0

        while not done:
            if obs not in Q:
                Q[obs] = np.zeros(2)
            if np.random.random() < eps:
                acc = np.random.randint(0, 2)
            else:
                acc = int(np.argmax(Q[obs]))

            obs_sig, reward, done = paso(estado, acc)
            if obs_sig not in Q:
                Q[obs_sig] = np.zeros(2)

            objetivo = reward
            if not done:
                objetivo += gamma * np.max(Q[obs_sig])
            Q[obs][acc] += lr * (objetivo - Q[obs][acc])

            obs = obs_sig
            total_reward += reward

        eps = max(eps_min, eps * eps_decay)
        rewards.append(total_reward)
        if total_reward > 0:
            wins += 1

        if (ep + 1) % 10000 == 0:
            avg = np.mean(rewards[-1000:])
            wr = wins / (ep + 1)
            print(f'Ep {ep+1}: reward={avg:.3f}, win_rate={wr:.3f}, estados={len(Q)}')

    return Q, rewards

In [ ]:
print('=== ENTRENANDO VEiumTUM CON Q-LEARNING ===')
Q, rewards = entrenar(episodios=50000)

## 3. Evaluacion del agente entrenado

In [ ]:
def evaluar(Q, episodios=10000):
    wins = draws = losses = 0
    for _ in range(episodios):
        estado, obs = reset()
        while True:
            if obs not in Q:
                Q[obs] = np.zeros(2)
            acc = int(np.argmax(Q[obs]))
            obs, reward, done = paso(estado, acc)
            if done:
                if reward > 0:
                    wins += 1
                elif reward == 0:
                    draws += 1
                else:
                    losses += 1
                break

    print(f'\nEvaluacion ({episodios} episodios):')
    print(f'  Victorias: {wins} ({100*wins/episodios:.1f}%)')
    print(f'  Empates:   {draws} ({100*draws/episodios:.1f}%)')
    print(f'  Derrotas:  {losses} ({100*losses/episodios:.1f}%)')

evaluar(Q)

## 4. Graficos de aprendizaje

In [ ]:
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.plot(rewards, alpha=0.3, label='Recompensa por episodio')
N = 1000
cumsum = np.cumsum(rewards)
avg = (cumsum[N-1:] - cumsum[:-N+1]) / N
plt.plot(avg, color='red', label=f'Media movil {N} ep.')
plt.xlabel('Episodio')
plt.ylabel('Recompensa')
plt.legend()
plt.title('Evolucion del aprendizaje')

plt.subplot(1, 2, 2)
acum_wins = np.cumsum(np.array(rewards) > 0)
win_rates = acum_wins / np.arange(1, len(rewards) + 1)
plt.plot(win_rates, label='Tasa de victorias')
plt.xlabel('Episodio')
plt.ylabel('Win rate')
plt.legend()
plt.title('Tasa de victorias acumulada')

plt.tight_layout()
plt.savefig('veiumtum_learning.png')
plt.show()

## 5. Jugar una partida manual

In [ ]:
def jugar_mano(Q):
    estado, obs = reset()
    print(f'\nTus cartas: {estado["jugador"]} (suma={valor_mano(estado["jugador"])})')
    print(f'Crupier muestra: {estado["crupier"][0]}')

    done = False
    while not done:
        if obs not in Q:
            Q[obs] = np.zeros(2)
        acc = int(np.argmax(Q[obs]))
        print(f'Agente decide: {"PEDIR" if acc == 1 else "PLANTARSE"}')
        obs, reward, done = paso(estado, acc)
        if not done or acc == 1:
            print(f'  Tus cartas: {estado["jugador"]} (suma={valor_mano(estado["jugador"])})')

    print(f'\nResultado final:')
    print(f'  Tus cartas: {estado["jugador"]} (suma={valor_mano(estado["jugador"])})')
    print(f'  Crupier:    {estado["crupier"]} (suma={valor_mano(estado["crupier"])})')
    if reward > 0:
        print('  >>> GANASTE!')
    elif reward == 0:
        print('  >>> EMPATE!')
    else:
        print('  >>> PERDISTE!')

jugar_mano(Q)